# Install and import packages

In [ ]:
%pip install -q -U langchain langchain-core langchain-groq langchain-community langchain-pinecone langchain-text-splitters pinecone pypdf spacy

In [ ]:
pip list

Package                               Version
------------------------------------- -------------------
absl-py                               2.0.0
accelerate                            1.15.0
access                                1.1.10.post3
affine                                3.0.1
aiofiles                              25.1.0
aiohappyeyeballs                      2.7.1
aiohttp                               3.14.5
aiohttp-retry                         2.9.1
aiosignal                             1.4.0
aiosqlite                             0.22.1
alabaster                             1.0.0
albucore                              0.0.24
albumentations                        2.0.8
ale-py                                0.12.1
altair                                5.5.0
annotated-doc                         0.0.5
annotated-types                       0.8.0
anyio                                 4.15.1
anywidget                             0.9.21
apsw                                  3.54.1.

In [ ]:
import os
import re
import torch
import time
import kagglehub

import pandas as pd

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

from sentence_transformers import SentenceTransformer

from groq import Groq
from groq.types import Model
from pinecone import Pinecone, ServerlessSpec

### Mount Drive and load API keys

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import userdata

for key in ("GROQ_API_KEY", "PINECONE_TOKEN"):
  os.environ[key] = userdata.get(key)

print("GROQ_API_KEY loaded:        ", bool(os.environ.get("GROQ_API_KEY")))
print("PINECONE_TOKEN loaded:", bool(os.environ.get("PINECONE_TOKEN")))

GROQ_API_KEY loaded:         True
PINECONE_TOKEN loaded: True


### Configuration

In [ ]:
DATASET_PATH_DRIVE = "/content/drive/MyDrive/games.csv"
PINECONE_INDEX_NAME = "obligatorio-tt2"
LLM_MODEL = "qwen/qwen3.8-27b"

# in case kagglehub is used
# DATASET_NAME = "fronkongames/steam-games-dataset"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# multilingual: questions in Spanish match game texts in English. Expects "query: " / "passage: " prefixes.
EMBEDDING_MODEL = "intfloat/multilingual-e5-base"

# measured in tokens of the embedding model: 350 + game name prefix + "passage: " stays below its 512 token limit
CHUNK_SIZE = 350
CHUNK_OVERLAP = 70
TOP_K = 8

# games missing any of these columns are discarded
REQUIRED_COLUMNS = ["Name", "About the game", "Reviews"]

# columns that go into the text that gets embedded (the rest is useless for semantic search)
TEXT_COLUMNS = ["Name", "Release date", "Price", "Genres", "Tags", "Categories", "Developers",
                "Publishers", "Supported languages", "About the game", "Reviews"]

# True deletes the Pinecone index and uploads every vector again
RESET_INDEX = False

### Check model and Pinecone Index

In [ ]:
def get_groq_models(models: list[Model]) -> list[str]:
    groq_models: list[str] = [model.id for model in models]
    return sorted(groq_models)

def check_chosen_model(available_models: list[str]) -> bool:
    if LLM_MODEL in available_models:
        print(f"Groq: valid key, {LLM_MODEL} model available")
        return True

    print(f"{LLM_MODEL} is not available on Groq. Available models: {available_models}")
    return False

# check if model is valid
available_models: list[str] = get_groq_models(Groq().models.list().data)
check_chosen_model(available_models)

# check pinecone indexes
pc: Pinecone = Pinecone(api_key=os.environ["PINECONE_TOKEN"])
index_names: list[str] = pc.list_indexes().names()
print("Pinecone: valid key, existing indexes:", index_names)

Groq: valid key, qwen/qwen3.8-27b model available
Pinecone: valid key, existing indexes: ['cinematchai-rag', 'obligatorio-tt2']


# Part 1

### Load dataset from CSV file and remove useless columns

In [ ]:
# https://www.kaggle.com/datasets/fronkongames/steam-games-dataset

df_raw = pd.read_csv(DATASET_PATH_DRIVE)

print(df_raw[["AppID", "Name", "Price", "Genres"]].head(3), "\n")

df = df_raw.drop(columns=['Screenshots', 'Movies', 'Header image', 'Support url',
                          'Metacritic url', 'User score', 'Score rank',
                          'Average playtime two weeks', 'Median playtime two weeks'])

print("Rows before removing games without", REQUIRED_COLUMNS, ":", len(df))
df = df.dropna(subset=REQUIRED_COLUMNS).reset_index(drop=True)
print("Rows after:", len(df))

# columns used as metadata to filter in Pinecone
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["release_year"] = pd.to_datetime(df["Release date"], format="mixed", errors="coerce").dt.year
total_reviews = df["Positive"] + df["Negative"]
df["positive_ratio"] = (df["Positive"] / total_reviews).where(total_reviews > 0)

display(df.head())

     AppID                                   Name  Price     Genres
0  2539430             Black Dragon Mage Playtest   0.00        NaN
1   496350  Supipara - Chapter 1 Spring Has Come!   5.24  Adventure
2  1034400      Mystery Solitaire The Black Raven   4.99     Casual 

Rows before removing games without ['Name', 'About the game', 'Reviews'] : 125855
Rows after: 12156


,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,Discount,DLC count,About the game,...,Notes,Average playtime forever,Median playtime forever,Developers,Publishers,Categories,Genres,Tags,release_year,positive_ratio
0,1025440,Fantasy General II,"Sep 5, 2019",100000 - 200000,13,0,13.99,0,4,Three hundred years have passed since the Shad...,...,NaN,489,52,Owned by Gravity,Slitherine Ltd.,"Single-player,Multi-player,PvP,Online PvP,Stea...",Strategy,"Strategy,Turn-Based Strategy,Fantasy,Turn-Base...",2019,0.829960
1,328460,Redline,"Oct 23, 2014",0 - 20000,0,0,1.04,0,0,Join the Company! It's 2066 and the world is a...,...,NaN,203,203,"Accolade, Inc.","Atari,Nightdive Studios","Single-player,Family Sharing","Action,Racing","Action,Racing,Gore,Violent,FPS,Classic,Post-ap...",2014,0.678899
2,1607040,Evertried,"Oct 21, 2021",0 - 20000,0,0,5.99,0,1,"'Every Move Matters' In Evertried, the player’...",...,NaN,480,480,Lunic Games / Danilo Domingues,DANGEN Entertainment,"Single-player,Full controller support,Family S...","Indie,Strategy","Strategy,Rogue-like,Mystery Dungeon,Rogue-lite...",2021,0.797101
3,688420,Bad North: Jotunn Edition,"Nov 16, 2018",500000 - 1000000,64,0,2.99,80,1,Position and relocate your troops to fend off ...,...,Bad North features fantasy blood and violence....,342,173,"Plausible Concept,Oskar Stålberg",Raw Fury,"Single-player,Steam Achievements,Full controll...","Action,Indie,Simulation","Wargame,Vikings,Tower Defense,Real Time Tactic...",2018,0.931874
4,2334500,Wizard Shrimp,"Dec 13, 2024",0 - 20000,1,0,2.99,0,0,"Wizard Shrimp™ is a bullet-hell, roguelite gam...",...,NaN,0,0,Wizard Shrimp Games,Wizard Shrimp Games,"Single-player,Steam Achievements,Full controll...","Action,Casual,Indie","Action Roguelike,Casual,Bullet Hell,Magic,Pixe...",2024,0.979381


### Build one text per game from the columns that carry meaning

In [ ]:
def has_value(value: object) -> bool:
    """True if the value is not NaN/None and not an empty or whitespace-only string."""
    return pd.notna(value) and str(value).strip() != ""


def game_to_text(row: pd.Series) -> str:
    """One "column: value" line per non-empty column of TEXT_COLUMNS."""
    lines: list[str] = []
    for col in TEXT_COLUMNS:
        value = row[col]
        if has_value(value):
            lines.append(f"{col}: {value}")

    return "\n".join(lines)


df["text"] = df.apply(game_to_text, axis=1)

In [ ]:
print("First game:\n", df.loc[0, "text"][:1500])
print("\nCharacters per game:", df["text"].str.len().describe()[["mean", "min", "max"]].round(0).to_dict())

First game:
 Name: Fantasy General II
Release date: Sep 5, 2019
Price: 13.99
Genres: Strategy
Tags: Strategy,Turn-Based Strategy,Fantasy,Turn-Based,Turn-Based Tactics,Turn-Based Combat,Tactical,Singleplayer,Wargame,Hex Grid,Grand Strategy,Dark Fantasy,4X,Multiplayer,War,Military,Replay Value,Resource Management,Great Soundtrack,Management
Categories: Single-player,Multi-player,PvP,Online PvP,Steam Achievements,Steam Trading Cards,Steam Cloud,Includes level editor,Family Sharing
Developers: Owned by Gravity
Publishers: Slitherine Ltd.
Supported languages: ['English', 'German', 'Spanish - Spain', 'Simplified Chinese', 'French', 'Russian']
About the game: Three hundred years have passed since the Shadow Wars have ravaged Keldonia and the world of Aer, and the struggles of the past have long since faded into legends. In the Highlands of Fareach, Clans of Barbarian warriors have eked out a living in the harsh northern climate, constantly feuding with each other and raiding the wealthier Bor

### Chunking

Most games don't fit in a single embedding (the model reads at most 512 tokens), so each game is split into chunks of `CHUNK_SIZE` tokens.
The size is measured with the tokenizer of the embedding model and not in characters: some games are described in Chinese, Japanese or Russian, where 1000 characters can be close to 800 tokens and would be truncated.
Chunks after the first one are prefixed with the game's name, so each chunk still says which game it belongs to.
Each chunk keeps the game's id, name, price, year, genres and positive ratio as metadata, used to filter searches and to attribute sources.

In [ ]:
from transformers import AutoTokenizer

# same tokenizer as the embedding model, so chunk sizes are measured the way the model reads them
tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)
splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, separators=["\n", ". ", " ", ""])


def split_list(value):
    return [s.strip() for s in str(value).split(",") if s.strip()] if pd.notna(value) else None


def optional(value, cast):
    return cast(value) if pd.notna(value) else None


chunks = pd.DataFrame([
    {"id": f"{game.AppID}-{i}", "app_id": str(game.AppID), "name": game.Name, "chunk": i,
     "text": text if i == 0 else f"Name: {game.Name}\n{text}",
     "price": optional(game.Price, float),
     "release_year": optional(game.release_year, int),
     "genres": split_list(game.Genres),
     "positive_ratio": optional(game.positive_ratio, float)}
    for game in df.itertuples()
    for i, text in enumerate(splitter.split_text(game.text))
])

print(f"{len(df)} games -> {len(chunks)} chunks ({len(chunks) / len(df):.1f} per game)")
print("Characters per chunk:", chunks["text"].str.len().describe()[["mean", "min", "max"]].round(0).to_dict())
print("\nExample (chunk 1 of the first game):\n" + chunks.loc[1, "text"][:500])

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (889 > 512). Running this sequence through the model will result in indexing errors


12156 games -> 41563 chunks (3.4 per game)
Characters per chunk: {'mean': 804.0, 'min': 32.0, 'max': 2021.0}

Example (chunk 1 of the first game):
Name: Fantasy General II
About the game: Three hundred years have passed since the Shadow Wars have ravaged Keldonia and the world of Aer, and the struggles of the past have long since faded into legends. In the Highlands of Fareach, Clans of Barbarian warriors have eked out a living in the harsh northern climate, constantly feuding with each other and raiding the wealthier Borderland towns. Fed up with these constant attacks, the Borderland Clans called on the Empire for help – a powerful realm


### Embedding model

In [ ]:
embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)

token_counts = pd.Series([len(ids) for ids in embedder.tokenizer(("passage: " + chunks["text"]).tolist())["input_ids"]])
print(f"Dimension: {embedder.get_embedding_dimension()} | max input length: {embedder.max_seq_length} tokens")
print(f"Tokens per chunk: mean {token_counts.mean():.0f}, max {token_counts.max()}, "
      f"truncated chunks: {(token_counts > embedder.max_seq_length).sum()}")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Dimension: 768 | max input length: 512 tokens
Tokens per chunk: mean 218, max 412, truncated chunks: 0


### Get embeddings and store them in Vector DB

In [ ]:
dimension = embedder.get_embedding_dimension()

# an index created with another embedding model has another dimension and can't be reused
if pc.has_index(PINECONE_INDEX_NAME) and (RESET_INDEX or pc.describe_index(PINECONE_INDEX_NAME).dimension != dimension):
    print("Deleting index:", PINECONE_INDEX_NAME)
    pc.delete_index(PINECONE_INDEX_NAME)

if not pc.has_index(PINECONE_INDEX_NAME):
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
index = pc.Index(PINECONE_INDEX_NAME)
print("Indexes:", pc.list_indexes().names())

Deleting index: obligatorio-tt2
Indexes: ['cinematchai-rag', 'obligatorio-tt2']


In [ ]:
# get embeddings from model ("passage: " prefix for documents, as e5 expects)
vectors = embedder.encode(chunks["text"].tolist(), prompt="passage: ", normalize_embeddings=True,
                          batch_size=64, show_progress_bar=True)

METADATA_COLUMNS = ["app_id", "name", "chunk", "text", "price", "release_year", "genres", "positive_ratio"]


def clean_metadata(chunk):
    """Pinecone rejects null values in metadata, so empty fields are left out."""
    return {col: chunk[col] for col in METADATA_COLUMNS if chunk[col] is not None and chunk[col] == chunk[col]}


records = [
    {"id": chunk["id"], "values": vector.tolist(), "metadata": clean_metadata(chunk)}
    for chunk, vector in zip(chunks.to_dict("records"), vectors)
]

# insert into the pinecone vector db (batches of 50: each request must weigh less than 2 MB)
index.upsert(vectors=records, batch_size=50)

# Pinecone serverless takes a few seconds for showing writes.
for _ in range(60):
    stats = index.describe_index_stats()
    if stats.total_vector_count == len(records):
        break
    time.sleep(2)
print(f"Index '{PINECONE_INDEX_NAME}': {stats.total_vector_count} vectors of {stats.dimension} dimensions")

Batches:   0%|          | 0/650 [00:00<?, ?it/s]

Upserted vectors:   0%|          | 0/41563 [00:00<?, ?it/s]

Index 'obligatorio-tt2': 41563 vectors of 768 dimensions


### Retrieval

In [ ]:
def retrieve(question, filters=None, k=TOP_K):
    """The k chunks most similar to the question, at most one per game, optionally filtered by metadata.

    filters example: {"genres": {"$in": ["RPG"]}, "price": {"$lte": 10}, "release_year": {"$gte": 2020}}
    """
    vector = embedder.encode(question, prompt="query: ", normalize_embeddings=True).tolist()
    response = index.query(vector=vector, top_k=k * 3, filter=filters, include_metadata=True)
    results, seen = [], set()
    for match in response.matches:
        if match.metadata["app_id"] not in seen:
            seen.add(match.metadata["app_id"])
            results.append({**match.metadata, "score": match.score})
    return results[:k]


# quick test
for r in retrieve("juegos de terror cooperativos", k=3):
    print(f'{r["score"]:.3f} | {r["name"]} | {r["text"][:150]!r}...')

0.831 | Outer Terror | 'Name: Outer Terror\nRelease date: Apr 20, 2023\nPrice: 9.99\nGenres: Action,Adventure,Indie\nTags: Action,Adventure,Action Roguelike,Action-Adventure,Arca'...
0.830 | Tricky Towers | 'Name: Tricky Towers\nRelease date: Aug 2, 2016\nPrice: 4.94\nGenres: Casual,Indie\nTags: Party Game,Puzzle,Multiplayer,Physics,Casual,Building,Funny,Famil'...
0.827 | Terrorformer TD | 'Name: Terrorformer TD\nRelease date: Sep 9, 2024\nPrice: 7.79\nGenres: Strategy\nTags: Tower Defense,Procedural Generation,Rogue-like,Rogue-lite,RTS,Strat'...


### Prompt Engineering

In [ ]:
NO_INFO = "I wasn't able to find enough information on the videogame catalog to answer that question."

SYSTEM_PROMPT = f"""You are an assistant specialized in Steam videogames. You answer questions using only the fragments of the game catalog you are given as context.

Rules:
1. Use only the information in the fragments. Do not add your own knowledge about games, studios, prices or release dates.
2. Cite every statement with the number of the fragment in square brackets, for example [2] or [1][4].
3. Each fragment starts with the name of the game it belongs to. Do not mix information from different games.
4. Name a game only if its name appears in a fragment. Do not make up titles, prices, dates or reviews.
5. Ignore the fragments that are not useful to answer, and do not cite them.
6. If no fragment answers the question, answer exactly: "{NO_INFO}" (translated to the language of the question).
7. The fragments are in English: always answer in the same language as the question, with an introductory sentence, a list of points and a short conclusion."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Catalog fragments:\n\n{context}\n\nQuestion: {question}"),
])

llm = ChatGroq(model=LLM_MODEL, temperature=0.2, max_tokens=800, reasoning_effort="none", max_retries=6)
rag_chain = prompt | llm | StrOutputParser()

### Asking questions (retrieval -> generation -> sources)

In [ ]:
def format_context(results):
    return "\n\n".join(f'[{n}] (game: {r["name"]}, app_id {r["app_id"]})\n{r["text"]}'
                       for n, r in enumerate(results, start=1))


def cited_sources(answer, n_results):
    """Numbers cited in the answer, like [2], [1][3] or [1, 3]."""
    groups = re.findall(r"\[([\d,\s]+)\]", answer)
    return sorted({int(n) for group in groups for n in re.findall(r"\d+", group) if 1 <= int(n) <= n_results})


def ask(question, filters=None, k=TOP_K):
    """RAG pipeline: retrieval -> generation -> source attribution."""
    results = retrieve(question, filters, k)

    print("=" * 110)
    print("QUESTION:", question)
    if filters:
        print("FILTERS: ", filters)
    print("=" * 110)
    print(f"\n--- RETRIEVED CONTEXT ({len(results)} fragments) ---")
    for n, r in enumerate(results, start=1):
        print(f'\n[{n}] score={r["score"]:.3f} | {r["name"]} (app_id {r["app_id"]}, chunk {int(r["chunk"])})')
        print("    " + r["text"][:300].replace("\n", " ") + "...")

    answer = rag_chain.invoke({"context": format_context(results), "question": question})
    answer = re.sub(r"<think>.*?</think>", "", answer, flags=re.S).strip()  # in case the model thinks out loud
    print("\n--- ANSWER ---")
    print(answer)

    cited = cited_sources(answer, len(results))
    print("\n--- SOURCES USED ---")
    if not cited:
        print("The answer doesn't cite any fragment.")
    for n in cited:
        r = results[n - 1]
        print(f'[{n}] {r["name"]} (app_id {r["app_id"]}, chunk {int(r["chunk"])}, score {r["score"]:.3f})')
    return {"question": question, "results": results, "answer": answer, "cited": cited}


outputs = {}

### Test queries

In [ ]:
outputs["Semantic"] = ask("Recomendame juegos cooperativos de supervivencia")

QUESTION: Recomendame juegos cooperativos de supervivencia

--- RETRIEVED CONTEXT (8 fragments) ---

[1] score=0.841 | How to Survive (app_id 250400, chunk 0)
    Name: How to Survive Release date: Aug 29, 2014 Price: 3.74 Genres: Action,Adventure,RPG Tags: Survival,Zombies,Crafting,Open World Survival Craft,Action,Co-op,Multiplayer,RPG,Local Co-Op,Online Co-Op,Adventure,Survival Horror,Open World,Hack and Slash,Post-apocalyptic,Gore,Singleplayer,Third Person...

[2] score=0.838 | Survivalist (app_id 340050, chunk 0)
    Name: Survivalist Release date: Jan 30, 2015 Price: 0.99 Genres: Action,Adventure,Indie,RPG,Simulation,Strategy Tags: Survival,Zombies,RPG,Open World,Action,Strategy,Base-Building,Adventure,Post-apocalyptic,Simulation,Multiplayer,Sandbox,Co-op,Singleplayer,RTS,Top-Down,Crafting,Action RPG,Open World...

[3] score=0.834 | Entropy Survivors (app_id 2602030, chunk 0)
    Name: Entropy Survivors Release date: Dec 4, 2024 Price: 3.99 Genres: Action,Casual,Indie,RPG Tags: Ac

In [ ]:
outputs["Semantic (English)"] = ask("Recommend me cooperative survival games")

QUESTION: Recommend me cooperative survival games

--- RETRIEVED CONTEXT (8 fragments) ---

[1] score=0.854 | How to Survive (app_id 250400, chunk 0)
    Name: How to Survive Release date: Aug 29, 2014 Price: 3.74 Genres: Action,Adventure,RPG Tags: Survival,Zombies,Crafting,Open World Survival Craft,Action,Co-op,Multiplayer,RPG,Local Co-Op,Online Co-Op,Adventure,Survival Horror,Open World,Hack and Slash,Post-apocalyptic,Gore,Singleplayer,Third Person...

[2] score=0.854 | Survive the Fall (app_id 1579340, chunk 4)
    Name: Survive the Fall Reviews: “A Post-Apocalyptic Masterclass in Story, Strategy, and Survival” (@razoreqx) “Survive the Fall combines some of the best bits of two of my favourite group survival games of the last ten years to impressive effect.” 8/10 – Loot Level Chill “We can definitely recommend...

[3] score=0.853 | Survivalist (app_id 340050, chunk 0)
    Name: Survivalist Release date: Jan 30, 2015 Price: 0.99 Genres: Action,Adventure,Indie,RPG,Simulation,Strategy 

In [ ]:
outputs["Filtered by genre and price"] = ask("¿Qué RPG baratos y bien valorados hay?", filters={"genres": {"$in": ["RPG"]}, "price": {"$lte": 10}})

QUESTION: ¿Qué RPG baratos y bien valorados hay?
FILTERS:  {'genres': {'$in': ['RPG']}, 'price': {'$lte': 10}}

--- RETRIEVED CONTEXT (8 fragments) ---

[1] score=0.823 | Endor Awakens: Roguelite DRPG (app_id 3594170, chunk 3)
    Name: Endor Awakens: Roguelite DRPG Reviews: “Un Rogue de manual, bien vitaminado de mecánicas y una buena cantidad de contenido. Un free to play respetuoso con el jugador y una monetización justa. Los amantes del género, en su forma más pura, tenéis aquí una propuesta de lo más atractiva.” DeVuego...

[2] score=0.807 | The I of the Dragon (app_id 279720, chunk 4)
    Name: The I of the Dragon Reviews: “Very good! [...] those of you looking for an Action RPG to play that has smooth and addictive game play and a unique twist to it should definitely pick up I of the Dragon.” 80 – Actiontrip “I'd definitely rank this as a 'must-buy' for any dragon-lovers out there [...

[3] score=0.807 | Blackguards (app_id 249650, chunk 0)
    Name: Blackguards Release date: Ja

In [ ]:
outputs["Filtered by year"] = ask("Juegos de terror lanzados desde 2020", filters={"release_year": {"$gte": 2020}})

QUESTION: Juegos de terror lanzados desde 2020
FILTERS:  {'release_year': {'$gte': 2020}}

--- RETRIEVED CONTEXT (8 fragments) ---

[1] score=0.851 | Fear the Spotlight (app_id 1959390, chunk 0)
    Name: Fear the Spotlight Release date: Oct 22, 2024 Price: 5.99 Genres: Action,Adventure,Indie Tags: Adventure,Survival Horror,Atmospheric,Story Rich,Puzzle,Stealth,Female Protagonist,Exploration,Horror,Psychological Horror,Third Person,Surreal,Action-Adventure,Retro,Action,3D,Stylized,Linear,Pixel ...

[2] score=0.846 | Layers of Fear (app_id 1946700, chunk 0)
    Name: Layers of Fear Release date: Jun 15, 2023 Price: 11.99 Genres: Adventure Tags: Psychological Horror,Horror,First-Person,Adventure,Story Rich,Psychedelic,Exploration,Atmospheric,Singleplayer,Multiple Endings,Dark,Mystery,Emotional,Immersive,Mature,Puzzle,Narrative,Interactive Fiction,Walking Si...

[3] score=0.845 | At Dead Of Night (app_id 1450830, chunk 0)
    Name: At Dead Of Night Release date: Nov 19, 2020 Price: 11.99 

### Hallucination test

The answer is not in the catalog: the model should say it doesn't have enough information.

In [ ]:
outputs["Out of catalog"] = ask("¿Quién ganó The Game Awards 2030?", k=5)

QUESTION: ¿Quién ganó The Game Awards 2030?

--- RETRIEVED CONTEXT (5 fragments) ---

[1] score=0.815 | The Game Awards (app_id 2819190, chunk 0)
    Name: The Game Awards Release date: Feb 27, 2024 Price: 0.99 Genres: Adventure,Casual,Indie Tags: Parody,Satire,Game Development,Narration,Dark Comedy,Cinematic,Action-Adventure,3D Platformer,Hack and Slash,Funny,Dark Humor,Exploration,Collectathon,Interactive Fiction,Platformer,3D,Third Person,Drag...

[2] score=0.772 | You Deserve (app_id 502240, chunk 2)
    Name: You Deserve Reviews: “I believe a fair rating would be an 8 out of 10 given the core elements of a horror adventure game at play, the atmospheric quality, and a somewhat interesting plot” 8/10 – Gaming Cypher “La estética de You Deserve, no obstante, tiene más de Silent Hill que de cualquier o...

[3] score=0.771 | System of Souls (app_id 2210130, chunk 0)
    Name: System of Souls Release date: Jun 8, 2023 Price: 1.94 Genres: Action,Indie Tags: Puzzle,Action,Puzzle-Platforme

# Part 2

### Load papers

In [ ]:
documents = []
routes = ["2507.14805v1.pdf", "Attention is all you need.pdf",
          "Designing Data-Intensive Applications The Big Ideas Behind Reliable, Scalable, and Maintainable Systems by Martin Kleppmann (z-l.pdf",
          "GPT Improving Language Understanding by Generative Pretraining.pdf",
          "Language Models are Few-Shot Learners (GPT3).pdf",
          "Scaling Laws for Neural Language Models.pdf"]

for route in routes:
  currentLoader = PyPDFLoader(
    file_path="/content/drive/MyDrive/Papers/" + route,
    mode = "single",
    extract_images=False
  )
  currentDocument = currentLoader.load()

  currentContent = currentDocument[0].page_content

  documents.append(currentContent)

print("Finished learning")

In [ ]:
for doc in documents:
  print("===================================== NEW DOC =====================================")
  print(doc.split("\n")[0])
  print(" ")